# 06 — VCR test harness

The VCR family (`vectors-vcr-core`, `vectors-vcr-junit5`, `vectors-vcr-testng`, `vectors-vcr-spring-ai`, `vectors-vcr-langchain4j`) is a record/replay harness for non-deterministic external calls — LLM completions, embedding APIs, anything else you would otherwise mock. On the first run, real responses are written to a "cassette"; subsequent runs replay from the cassette and never hit the network.

This notebook shows the bare-metal `CassetteStore` SPI directly — no framework plumbing — so you can see exactly what gets written and read. For production test code, use `@VCRTest` (JUnit 5) or `@VCRTestNG` (TestNG); see `:demos:vcr-e2e`.

In [1]:
%classpath /home/jovyan/work/vectors/vectors-core/build/libs/vectors-core-0.1.0-SNAPSHOT.jar
%classpath /home/jovyan/work/vectors/vectors-storage/build/libs/vectors-storage-0.1.0-SNAPSHOT.jar
%classpath /home/jovyan/work/vectors/vectors-vcr-core/build/libs/vectors-vcr-core-0.1.0-SNAPSHOT.jar
%classpath /home/jovyan/work/vectors/vectors-vcr-serde-jackson/build/libs/vectors-vcr-serde-jackson-0.1.0-SNAPSHOT.jar


In [2]:
%%loadFromPOM
<dependencies>
  <dependency>
    <groupId>com.fasterxml.jackson.core</groupId>
    <artifactId>jackson-databind</artifactId>
    <version>2.18.2</version>
  </dependency>
</dependencies>


In [3]:
import com.integrallis.vectors.storage.backend.LocalFileStorageBackend;
import com.integrallis.vectors.vcr.CassetteKey;
import com.integrallis.vectors.vcr.CassetteRecord;
import com.integrallis.vectors.vcr.ExactCassetteStore;
import java.nio.file.Files;
import java.nio.file.Path;
import java.util.Optional;

Path cassetteDir = Files.createTempDirectory("vcr-notebook-");
LocalFileStorageBackend backend = new LocalFileStorageBackend(cassetteDir);
ExactCassetteStore store = new ExactCassetteStore(backend);
System.out.println("cassette dir: " + cassetteDir);

cassette dir: /tmp/vcr-notebook-1088346439143402025


In [4]:
// --- Record phase: simulate a real LLM call and stash the response.
CassetteKey key = new CassetteKey("embedding", "notebook-06", 0);
CassetteRecord.Embedding recorded = new CassetteRecord.Embedding(
    "notebook-06",
    "text-embedding-3-small",
    System.currentTimeMillis(),
    new float[] {0.11f, 0.22f, 0.33f, 0.44f});
store.store(key, recorded);
System.out.println("recorded under key: " + key.serializedKey());

// Peek at what landed on disk:
try (var walk = Files.walk(cassetteDir)) {
    walk.filter(Files::isRegularFile).forEach(p ->
        System.out.printf("  %s  (%d bytes)%n", cassetteDir.relativize(p), p.toFile().length()));
}

recorded under key: vcr:embedding:notebook-06:0000


vcr:embedding:notebook-06:0000.etag

  (

12

 bytes)

vcr:embedding:notebook-06:0000

  (

136

 bytes)

In [5]:
// --- Replay phase: read back the record. In a real test, the model wrapper
//     would call this before hitting the network and return the cached value.
Optional<CassetteRecord> replayed = store.retrieve(key);
if (replayed.isPresent() && replayed.get() instanceof CassetteRecord.Embedding e) {
    System.out.printf("model=%s  recordedAt=%d  vec[0]=%f%n",
        e.model(), e.timestamp(), e.embedding()[0]);
}

System.out.println("store.listByTestId('notebook-06') -> " + store.listByTestId("notebook-06"));

model=

text-embedding-3-small

  recordedAt=

1777066721520

  vec[0]=

0.110000

store.listByTestId('notebook-06') -> [CassetteKey[type=embedding, testId=notebook-06, callIndex=0]]


In [6]:
// Cleanup — ExactCassetteStore holds no resources itself; just walk the cassette dir and delete.
try (var walk = Files.walk(cassetteDir)) {
    walk.sorted((a, b) -> b.getNameCount() - a.getNameCount()).forEach(p -> p.toFile().delete());
}
System.out.println("removed " + cassetteDir);

removed /tmp/vcr-notebook-1088346439143402025


## In a real test

```java
@VCRTest(mode = VCRMode.PLAYBACK_OR_RECORD, cassettePath = "src/test/resources/cassettes")
class MyTest {
    @VCRModel(modelName = "text-embedding-3-small")
    EmbeddingModel embeddings = OpenAiEmbeddingModel.builder().apiKey(System.getenv("OPENAI_API_KEY")).build();

    @Test void firstRunRecords_subsequentRunsReplay() {
        var e = embeddings.embed("hello world");
        assertThat(e.content().vector()).hasSize(1536);
    }
}
```

The extension transparently wraps the `EmbeddingModel` / `ChatLanguageModel` / Spring AI counterparts with the corresponding VCR adapter and flips between `PLAYBACK` and `RECORD` based on the cassette state. See `:demos:vcr-e2e` for a JUnit 5 test class that exercises the whole stack.